**V1 experiment.** This notebook contains the V1 implementation and experiment configuration. See [methods and evaluation](../../docs/METHODS.md) for the implemented protocol.


In [ ]:
import sys
import time
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import build_knn_graph, label_propagation
from v2.utils.splits import stratified_clip_mask
from v2.utils.runs import make_run_dir

LEVEL_DIR = PROJECT_ROOT / "v2" / "Level2"
DATA_DIR = LEVEL_DIR / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Per-run output folders. A single notebook execution creates one runs/runN
# that holds the figures + results from Fix 1 + Fix 5 + Fix 6 (the three
# script blocks below all share the same RUN_DIR). Edit the notebook
# between runs to compare configurations.
RUN_DIR = make_run_dir(LEVEL_DIR, notebook_label="level2_classification_v2.ipynb")
FIG_DIR = RUN_DIR / "figures"
RES_DIR = RUN_DIR / "results"
print(f"this run -> {RUN_DIR}")

V1_DATA = PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz"


## Load Level 2 features (V1 .npz, original 51-dim with chroma)


In [ ]:
data = np.load(V1_DATA, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["labels"].astype(np.int64)
clip_ids = data["clip_ids"].astype(np.int64)
label_names = list(data["label_names"])
is_train = data["is_train"]

print(f"X: {X.shape}, classes: {len(label_names)}, clips: {len(np.unique(clip_ids))}")

scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)

W = build_knn_graph(Xz, k=10, sigma="median")
print(f"W: {W.shape}, nnz={W.nnz}")


## Sweep label fraction at clip-level split (default k=10, σ=median)


In [ ]:
N_TRIALS = 20
LABEL_FRACS = [0.30, 0.50, 0.75]

def run_trial(W, mask):
    pred = label_propagation(W, y, mask)
    unl = ~mask
    return (
        accuracy_score(y[unl], pred[unl]),
        f1_score(y[unl], pred[unl], average="macro",
                 labels=range(len(label_names)), zero_division=0),
        pred,
    )

sweep = {}
all_masks = {}
all_preds_first_trial = {}

t0 = time.time()
for frac in LABEL_FRACS:
    accs, f1s, masks = [], [], []
    first_pred = None
    for t in range(N_TRIALS):
        rng = np.random.default_rng(3000 + int(frac * 100) * 100 + t)
        m = stratified_clip_mask(clip_ids, y, frac, rng)
        a, f, p = run_trial(W, m)
        accs.append(a); f1s.append(f); masks.append(m)
        if t == 0:
            first_pred = p
    sweep[frac] = {
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs)),
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
    }
    all_masks[frac] = masks
    all_preds_first_trial[frac] = first_pred
    print(f"  frac={frac:.2f}: acc {sweep[frac]['acc_mean']:.4f} ± {sweep[frac]['acc_std']:.4f}, "
          f"macro F1 {sweep[frac]['f1_mean']:.4f} ± {sweep[frac]['f1_std']:.4f}")

print(f"\nLevel 2 V2 sweep done in {time.time() - t0:.1f}s")


## Reference: V1 window-level 50% setting (re-run for an apples-to-apples Î”)


In [ ]:
def stratified_window_mask(y, frac, rng):
    mask = np.zeros(y.shape[0], dtype=bool)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        n = max(1, int(round(frac * idx.size)))
        chosen = rng.choice(idx, size=n, replace=False)
        mask[chosen] = True
    return mask

ref_accs, ref_f1s = [], []
for t in range(N_TRIALS):
    rng = np.random.default_rng(1000 + t)
    m = stratified_window_mask(y, 0.5, rng)
    a, f, _ = run_trial(W, m)
    ref_accs.append(a); ref_f1s.append(f)
print(f"V1 window-split 50% (reference): acc {np.mean(ref_accs):.4f} ± {np.std(ref_accs):.4f}, "
      f"macro F1 {np.mean(ref_f1s):.4f} ± {np.std(ref_f1s):.4f}")


## Persist results + per-fraction trial-0 masks for downstream CNN baselines


In [ ]:
results = {
    "graph_params": {"k": 10, "sigma": "median"},
    "n_trials": N_TRIALS,
    "v1_window_split_50pct": {
        "accuracy_mean": float(np.mean(ref_accs)),
        "accuracy_std": float(np.std(ref_accs)),
        "f1_mean": float(np.mean(ref_f1s)),
        "f1_std": float(np.std(ref_f1s)),
    },
    "v2_clip_split": {f"{int(frac*100)}pct": sweep[frac] for frac in LABEL_FRACS},
}
out_json = RES_DIR / "level2_results_v2_fix1.json"
out_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
print("wrote", out_json)

mask_save = {f"frac_{int(frac*100)}_trial0": all_masks[frac][0] for frac in LABEL_FRACS}
mask_save["clip_ids"] = clip_ids
mask_save["is_train"] = is_train
# Snapshot inside this run + cache to Data/ (latest wins) for the resnet
# notebook to pick up.
np.savez_compressed(RES_DIR / "level2_v2_clip_masks.npz", **mask_save)
np.savez_compressed(DATA_DIR / "level2_v2_clip_masks.npz", **mask_save)
print("wrote masks to", RES_DIR / "level2_v2_clip_masks.npz")

full_save = {f"frac_{int(frac*100)}": np.stack(all_masks[frac]) for frac in LABEL_FRACS}
np.savez_compressed(RES_DIR / "level2_v2_clip_masks_all_trials.npz", **full_save)
np.savez_compressed(DATA_DIR / "level2_v2_clip_masks_all_trials.npz", **full_save)
print("wrote", RES_DIR / "level2_v2_clip_masks_all_trials.npz")


# Level 2 V2 — Fix 5: graph connectivity + parameter sweep at clip-level split

V1's best graph (k=3, σ=self_tuning) had 2 connected components: a main
component of 2787 nodes and a tiny isolated component of 6 nodes. Label
propagation cannot define f_u for the isolated component (the L_uu block
is rank-deficient on those rows), so the prediction there is essentially
random. Diffusion maps also collapse near this regime.

Fix 5:
  - Add `v2/utils/graph.py::ensure_connected(W, X)` (already shipped in
    Fix 1 commit). It splices in weak edges (fallback weight = 1e-6)
    between the closest pair of nodes in any disconnected components.
  - Re-run the V1 parameter sweep at clip-level 50% labels, applying the
    connectivity fix after each k-NN build.
  - Report `n_components` per (k, σ) so it's clear where the fragility
    was in V1.
  - After selecting best (k, σ), redo the label-fraction sweep
    (30 / 50 / 75%) at the best parameters.


In [ ]:
import sys
import time
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import build_knn_graph, label_propagation, spectral_gap_from_W
from v2.utils.graph import ensure_connected, n_connected_components
from v2.utils.splits import stratified_clip_mask
from utils.evaluation import plot_parameter_heatmap
from v2.utils.runs import make_run_dir

LEVEL_DIR = PROJECT_ROOT / "v2" / "Level2"
DATA_DIR = LEVEL_DIR / "Data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# When this script is concatenated into level2_classification_v2.ipynb the
# Fix 1 block above already created RUN_DIR / FIG_DIR / RES_DIR. When run
# standalone, create them here.
try:
    RUN_DIR  # type: ignore[name-defined]
except NameError:
    RUN_DIR = make_run_dir(LEVEL_DIR, notebook_label="level2_classification_v2.ipynb (fix5 standalone)")
    FIG_DIR = RUN_DIR / "figures"
    RES_DIR = RUN_DIR / "results"
    print(f"this run -> {RUN_DIR}")

V1_DATA = PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz"


## Load V1 features (Fix 6 swaps these out for the chroma-replaced set)


In [ ]:
data = np.load(V1_DATA, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["labels"].astype(np.int64)
clip_ids = data["clip_ids"].astype(np.int64)
label_names = list(data["label_names"])
is_train = data["is_train"]

scaler = StandardScaler()
Xz = scaler.fit_transform(X).astype(np.float32)
print(f"X: {X.shape}, classes: {len(label_names)}, clips: {len(np.unique(clip_ids))}")


## Helper: clip-level trial running with connectivity fix


In [ ]:
def run_trial(W, mask):
    pred = label_propagation(W, y, mask)
    unl = ~mask
    return (
        accuracy_score(y[unl], pred[unl]),
        f1_score(y[unl], pred[unl], average="macro",
                 labels=range(len(label_names)), zero_division=0),
        pred,
    )

def sigma_for(k, label, Xz):
    if label == "self_tuning":
        return "self_tuning"
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Xz)
    d, _ = nn.kneighbors(Xz)
    med = float(np.median(d[:, 1:]))
    factor = float(label.split("xmed")[0])
    return factor * med


## Parameter sweep (k, σ) at clip-level 50% labels, with connectivity fix

6 k × 5 σ × 20 trials = 600 graph-SSL runs. The k-NN graph is built once
per (k, σ) cell, then `ensure_connected` is applied. Both before- and
after-fix component counts are recorded.


In [ ]:
N_TRIALS = 20
LABEL_FRAC = 0.5
K_LIST = [3, 5, 7, 10, 15, 20]
SIGMA_LIST = ["0.5xmed", "1.0xmed", "2.0xmed", "5.0xmed", "self_tuning"]

# Pre-generate clip masks (shared across (k, σ) cells for fair averaging).
trial_masks = [
    stratified_clip_mask(clip_ids, y, LABEL_FRAC, np.random.default_rng(5000 + t))
    for t in range(N_TRIALS)
]
print(f"trial masks: {N_TRIALS} clip-level masks at frac={LABEL_FRAC}")

acc_grid = np.zeros((len(SIGMA_LIST), len(K_LIST)))
f1_grid = np.zeros_like(acc_grid)
gap_grid = np.zeros_like(acc_grid)
ncomp_pre = np.zeros((len(SIGMA_LIST), len(K_LIST)), dtype=int)
ncomp_post = np.zeros((len(SIGMA_LIST), len(K_LIST)), dtype=int)

t0 = time.time()
for i, slabel in enumerate(SIGMA_LIST):
    for j, k in enumerate(K_LIST):
        sig = sigma_for(k, slabel, Xz)
        W_raw = build_knn_graph(Xz, k=k, sigma=sig)
        ncomp_pre[i, j] = n_connected_components(W_raw)
        W = ensure_connected(W_raw, Xz, fallback_weight=1e-6)
        ncomp_post[i, j] = n_connected_components(W)

        eigs = spectral_gap_from_W(W, n_eigenvalues=10)
        diffs = np.diff(eigs)
        gap_grid[i, j] = float(diffs.max() if diffs.size else 0.0)

        accs, f1s = [], []
        for mask in trial_masks:
            a, f, _ = run_trial(W, mask)
            accs.append(a); f1s.append(f)
        acc_grid[i, j] = float(np.mean(accs))
        f1_grid[i, j] = float(np.mean(f1s))
        print(f"  k={k:>2d} sigma={slabel:<12s} | comp pre→post = {ncomp_pre[i,j]}→{ncomp_post[i,j]} | "
              f"acc={acc_grid[i,j]:.4f} F1={f1_grid[i,j]:.4f} gap={gap_grid[i,j]:.4f}")

print(f"\nFix 5 sweep done in {time.time() - t0:.1f}s")


## Where was V1 disconnected?


In [ ]:
print("\nDisconnected (k, σ) cells before connectivity fix:")
disc = []
for i, slabel in enumerate(SIGMA_LIST):
    for j, k in enumerate(K_LIST):
        if ncomp_pre[i, j] > 1:
            print(f"  k={k:>2d} sigma={slabel:<12s} : {ncomp_pre[i, j]} components")
            disc.append((slabel, k, int(ncomp_pre[i, j])))
print(f"After fix, all cells: {ncomp_post.max()} component(s)")

best_i, best_j = np.unravel_index(int(np.argmax(acc_grid)), acc_grid.shape)
best_sigma = SIGMA_LIST[best_i]
best_k = K_LIST[best_j]
best_acc = float(acc_grid[best_i, best_j])
print(f"\nBest (k, σ) at clip-level 50%: k={best_k}, σ={best_sigma}, "
      f"acc={best_acc:.4f}")


## Save sweep heatmaps


In [ ]:
plot_parameter_heatmap(K_LIST, SIGMA_LIST, acc_grid,
                       FIG_DIR / "fix5_param_heatmap_acc_v2.png",
                       title="Level 2 V2 — accuracy over (k, σ), clip-level 50%, connectivity fix")
plot_parameter_heatmap(K_LIST, SIGMA_LIST, ncomp_pre.astype(float),
                       FIG_DIR / "fix5_param_heatmap_ncomp_pre_v2.png",
                       title="Level 2 V2 — n_components pre-fix")


## Label fraction sweep at best (k, σ) with connectivity fix


In [ ]:
sig_best = sigma_for(best_k, best_sigma, Xz)
W_best_raw = build_knn_graph(Xz, k=best_k, sigma=sig_best)
W_best = ensure_connected(W_best_raw, Xz, fallback_weight=1e-6)
print(f"best graph: pre={n_connected_components(W_best_raw)} comp, "
      f"post={n_connected_components(W_best)} comp, "
      f"nnz={W_best.nnz}")

LABEL_FRACS = [0.30, 0.50, 0.75]
sweep_best = {}
all_masks_best = {}
for frac in LABEL_FRACS:
    accs, f1s, masks = [], [], []
    for t in range(N_TRIALS):
        rng = np.random.default_rng(7000 + int(frac * 100) * 100 + t)
        m = stratified_clip_mask(clip_ids, y, frac, rng)
        a, f, _ = run_trial(W_best, m)
        accs.append(a); f1s.append(f); masks.append(m)
    sweep_best[frac] = {
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs)),
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
    }
    all_masks_best[frac] = masks
    print(f"  frac={frac:.2f} (best params): "
          f"acc {sweep_best[frac]['acc_mean']:.4f} ± {sweep_best[frac]['acc_std']:.4f}, "
          f"macro F1 {sweep_best[frac]['f1_mean']:.4f} ± {sweep_best[frac]['f1_std']:.4f}")


## Persist Fix 5 results


In [ ]:
results_fix5 = {
    "trial_count": N_TRIALS,
    "param_sweep_label_frac_clip_level": LABEL_FRAC,
    "k_list": K_LIST,
    "sigma_list": SIGMA_LIST,
    "acc_grid": acc_grid.tolist(),
    "f1_grid": f1_grid.tolist(),
    "gap_grid": gap_grid.tolist(),
    "n_components_pre_fix": ncomp_pre.tolist(),
    "n_components_post_fix": ncomp_post.tolist(),
    "disconnected_pre_fix": disc,
    "best_params": {"k": int(best_k), "sigma_label": best_sigma, "acc": best_acc},
    "label_sweep_at_best_params": {
        f"{int(f*100)}pct": sweep_best[f] for f in LABEL_FRACS
    },
}
out_json = RES_DIR / "level2_results_v2_fix5.json"
out_json.write_text(json.dumps(results_fix5, indent=2), encoding="utf-8")
print("wrote", out_json)

mask_save = {f"frac_{int(f*100)}_trial0": all_masks_best[f][0] for f in LABEL_FRACS}
np.savez_compressed(RES_DIR / "level2_v2_clip_masks_bestparams.npz", **mask_save)
np.savez_compressed(DATA_DIR / "level2_v2_clip_masks_bestparams.npz", **mask_save)
print("wrote masks at best params to", RES_DIR)


# Level 2 V2 — Fix 6: chroma replaced with MFCC/dMFCC variability

Re-runs everything from Fix 5 (parameter sweep + label fraction sweep at
best params, with the connectivity correction) on the V2 features:

  13 MFCC mean + 13 dMFCC mean + 6 spectral stats + 7 spectral contrast
  + **6 MFCC std + 6 dMFCC std**  (replaces 12 chroma — same 51 dims).

We then publish the final accuracy-vs-labels plot using the V2 features
at best params, alongside the ResNet-18 fair / full lines from Fix 2.


In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.graph import build_knn_graph, label_propagation, spectral_gap_from_W
from v2.utils.graph import ensure_connected, n_connected_components
from v2.utils.splits import stratified_clip_mask
from utils.evaluation import plot_parameter_heatmap, plot_confusion_matrix
from v2.utils.runs import make_run_dir, find_latest_result

LEVEL_DIR = PROJECT_ROOT / "v2" / "Level2"
DATA_DIR = LEVEL_DIR / "Data"

# Inherit RUN_DIR / FIG_DIR / RES_DIR if Fix 1 / Fix 5 already ran above in
# the same notebook; otherwise create a fresh standalone run.
try:
    RUN_DIR  # type: ignore[name-defined]
except NameError:
    RUN_DIR = make_run_dir(LEVEL_DIR, notebook_label="level2_classification_v2.ipynb (fix6 standalone)")
    FIG_DIR = RUN_DIR / "figures"
    RES_DIR = RUN_DIR / "results"
    print(f"this run -> {RUN_DIR}")


## Load V2 features (chroma replaced)


In [ ]:
data = np.load(DATA_DIR / "level2_features_v2.npz", allow_pickle=True)
X_v2 = data["X"].astype(np.float32)
y = data["labels"].astype(np.int64)
clip_ids = data["clip_ids"].astype(np.int64)
label_names = list(data["label_names"])
is_train = data["is_train"]

scaler = StandardScaler()
Xz_v2 = scaler.fit_transform(X_v2).astype(np.float32)
print(f"V2 features: {X_v2.shape}, classes: {len(label_names)}")

# Quick comparison: V1 vs V2 features at the same default params, same trial set.
data_v1 = np.load(PROJECT_ROOT / "Level2" / "Data" / "level2_features.npz", allow_pickle=True)
X_v1 = data_v1["X"].astype(np.float32)
Xz_v1 = StandardScaler().fit_transform(X_v1).astype(np.float32)


## Helpers


In [ ]:
def run_trial(W, mask):
    pred = label_propagation(W, y, mask)
    unl = ~mask
    return (
        accuracy_score(y[unl], pred[unl]),
        f1_score(y[unl], pred[unl], average="macro",
                 labels=range(len(label_names)), zero_division=0),
        pred,
    )

def sigma_for(k, label, Xz):
    if label == "self_tuning":
        return "self_tuning"
    nn = NearestNeighbors(n_neighbors=k + 1).fit(Xz)
    d, _ = nn.kneighbors(Xz)
    med = float(np.median(d[:, 1:]))
    factor = float(label.split("xmed")[0])
    return factor * med


## Direct V1 vs V2 feature comparison at default (k=10, σ=median), 50% labels


In [ ]:
N_TRIALS = 20
LABEL_FRAC = 0.5
shared_masks = [
    stratified_clip_mask(clip_ids, y, LABEL_FRAC, np.random.default_rng(8000 + t))
    for t in range(N_TRIALS)
]

W_v1_default = ensure_connected(build_knn_graph(Xz_v1, k=10, sigma="median"), Xz_v1)
W_v2_default = ensure_connected(build_knn_graph(Xz_v2, k=10, sigma="median"), Xz_v2)

v1_default = [run_trial(W_v1_default, m)[:2] for m in shared_masks]
v2_default = [run_trial(W_v2_default, m)[:2] for m in shared_masks]
v1_acc = np.array([a for a, _ in v1_default])
v2_acc = np.array([a for a, _ in v2_default])
print(f"V1 features default (k=10, σ=median): acc {v1_acc.mean():.4f} ± {v1_acc.std():.4f}")
print(f"V2 features default (k=10, σ=median): acc {v2_acc.mean():.4f} ± {v2_acc.std():.4f}")
print(f"Î” (V2 − V1): {(v2_acc.mean() - v1_acc.mean()):+.4f}")


## Parameter sweep on V2 features with connectivity fix at clip-level 50%


In [ ]:
K_LIST = [3, 5, 7, 10, 15, 20]
SIGMA_LIST = ["0.5xmed", "1.0xmed", "2.0xmed", "5.0xmed", "self_tuning"]

acc_grid = np.zeros((len(SIGMA_LIST), len(K_LIST)))
f1_grid = np.zeros_like(acc_grid)
gap_grid = np.zeros_like(acc_grid)
ncomp_pre = np.zeros_like(acc_grid, dtype=int)
ncomp_post = np.zeros_like(acc_grid, dtype=int)

t0 = time.time()
for i, slabel in enumerate(SIGMA_LIST):
    for j, k in enumerate(K_LIST):
        sig = sigma_for(k, slabel, Xz_v2)
        Wraw = build_knn_graph(Xz_v2, k=k, sigma=sig)
        ncomp_pre[i, j] = n_connected_components(Wraw)
        W = ensure_connected(Wraw, Xz_v2, fallback_weight=1e-6)
        ncomp_post[i, j] = n_connected_components(W)

        eigs = spectral_gap_from_W(W, n_eigenvalues=10)
        diffs = np.diff(eigs)
        gap_grid[i, j] = float(diffs.max() if diffs.size else 0.0)

        accs, f1s = [], []
        for mask in shared_masks:
            a, f, _ = run_trial(W, mask)
            accs.append(a); f1s.append(f)
        acc_grid[i, j] = float(np.mean(accs))
        f1_grid[i, j] = float(np.mean(f1s))
        print(f"  k={k:>2d} sigma={slabel:<12s} | comp pre={ncomp_pre[i,j]} | "
              f"acc={acc_grid[i,j]:.4f} F1={f1_grid[i,j]:.4f}")
print(f"\nFix 6 sweep done in {time.time() - t0:.1f}s")

best_i, best_j = np.unravel_index(int(np.argmax(acc_grid)), acc_grid.shape)
best_sigma_v2 = SIGMA_LIST[best_i]
best_k_v2 = K_LIST[best_j]
best_acc_v2 = float(acc_grid[best_i, best_j])
print(f"\nV2 features: best (k, σ) at clip-level 50% = "
      f"k={best_k_v2}, σ={best_sigma_v2}, acc={best_acc_v2:.4f}")

plot_parameter_heatmap(K_LIST, SIGMA_LIST, acc_grid,
                       FIG_DIR / "fix6_param_heatmap_acc_v2_features.png",
                       title="Level 2 V2 features — accuracy over (k, σ), clip 50%")


## Label-fraction sweep at best (k, σ) on V2 features


In [ ]:
sig_best = sigma_for(best_k_v2, best_sigma_v2, Xz_v2)
W_best = ensure_connected(build_knn_graph(Xz_v2, k=best_k_v2, sigma=sig_best), Xz_v2)

LABEL_FRACS = [0.30, 0.50, 0.75]
sweep_v2_best = {}
masks_v2_best = {}
for frac in LABEL_FRACS:
    accs, f1s, masks = [], [], []
    for t in range(N_TRIALS):
        rng = np.random.default_rng(9000 + int(frac * 100) * 100 + t)
        m = stratified_clip_mask(clip_ids, y, frac, rng)
        a, f, _ = run_trial(W_best, m)
        accs.append(a); f1s.append(f); masks.append(m)
    sweep_v2_best[frac] = {
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs)),
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
    }
    masks_v2_best[frac] = masks
    print(f"  frac={frac:.2f} V2 best: "
          f"acc {sweep_v2_best[frac]['acc_mean']:.4f} ± {sweep_v2_best[frac]['acc_std']:.4f}, "
          f"F1 {sweep_v2_best[frac]['f1_mean']:.4f} ± {sweep_v2_best[frac]['f1_std']:.4f}")


## Confusion matrix at trial 0, best params, 50% labels


In [ ]:
mask0 = masks_v2_best[0.50][0]
unl0 = ~mask0
_, _, pred0 = run_trial(W_best, mask0)
plot_confusion_matrix(
    y[unl0], pred0[unl0], label_names,
    save_path=FIG_DIR / "fix6_confusion_matrix_v2_best.png",
    title=f"Level 2 V2 — label prop, V2 features, k={best_k_v2}, σ={best_sigma_v2}, 50% clip-level",
)


## Final accuracy-vs-labels plot (graph SSL @ default and @ best, vs CNN)


In [ ]:
# Read latest run-folder snapshots; if Fix 1/5 ran in this same notebook
# they live in this run's results/ dir, otherwise we fall back to the last
# stored run.
def _load_latest(level_dir, filename):
    p = find_latest_result(level_dir, filename)
    if p is None:
        raise FileNotFoundError(
            f"No run produced {filename} under {level_dir}; "
            f"run the corresponding notebook first."
        )
    return json.loads(p.read_text(encoding="utf-8")), p

gss_fix1, p1 = _load_latest(LEVEL_DIR, "level2_results_v2_fix1.json")
gss_fix5, p5 = _load_latest(LEVEL_DIR, "level2_results_v2_fix5.json")
cnn_results, pc = _load_latest(LEVEL_DIR, "resnet_v2_results.json")
print(f"  fix1 from: {p1}")
print(f"  fix5 from: {p5}")
print(f"  cnn  from: {pc}")
cnn_by_name = {r["name"]: r for r in cnn_results["runs"]}

fracs = [0.30, 0.50, 0.75]

gss_default_acc = [gss_fix1["v2_clip_split"][f"{int(f*100)}pct"]["acc_mean"] for f in fracs]
gss_default_std = [gss_fix1["v2_clip_split"][f"{int(f*100)}pct"]["acc_std"] for f in fracs]
gss_best_v1feat = [gss_fix5["label_sweep_at_best_params"][f"{int(f*100)}pct"]["acc_mean"] for f in fracs]
gss_best_v1feat_std = [gss_fix5["label_sweep_at_best_params"][f"{int(f*100)}pct"]["acc_std"] for f in fracs]
gss_best_v2feat = [sweep_v2_best[f]["acc_mean"] for f in fracs]
gss_best_v2feat_std = [sweep_v2_best[f]["acc_std"] for f in fracs]

cnn_fair = [cnn_by_name[f"L2-fair-{int(f*100)}"]["final_acc"] for f in fracs]
cnn_full = cnn_by_name["L2-full"]["final_acc"]

fig, ax = plt.subplots(figsize=(9, 6))
xs = [int(f*100) for f in fracs]
ax.errorbar(xs, gss_default_acc, yerr=gss_default_std, marker="o", capsize=3,
            label="Graph SSL (V1 feats, default k=10/σ=med)")
ax.errorbar(xs, gss_best_v1feat, yerr=gss_best_v1feat_std, marker="^", capsize=3,
            label=f"Graph SSL (V1 feats, best Fix 5: k=5/σ=self_tune)")
ax.errorbar(xs, gss_best_v2feat, yerr=gss_best_v2feat_std, marker="D", capsize=3, linewidth=2,
            label=f"Graph SSL (V2 feats, best k={best_k_v2}/σ={best_sigma_v2})")
ax.plot(xs, cnn_fair, marker="s", linestyle="-", label="ResNet-18 (50 ep, same clip-level labels)")
ax.axhline(cnn_full, color="tab:red", linestyle="--",
           label=f"ResNet-18 (100% HF train, 50 ep) = {cnn_full:.3f}")
ax.set_xlabel("label fraction (clip-level), %")
ax.set_ylabel("test accuracy")
ax.set_title("Level 2 V2 — accuracy vs label fraction (final)")
ax.set_xticks(xs)
ax.grid(True, alpha=0.3); ax.legend(loc="lower right", fontsize=8.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "fix6_accuracy_vs_labels_final.png", dpi=150)
plt.close(fig)
print("saved", FIG_DIR / "fix6_accuracy_vs_labels_final.png")


## Persist Fix 6 results


In [ ]:
results_fix6 = {
    "feature_set": "V2 (chroma swapped for 6 MFCC std + 6 dMFCC std)",
    "v1_vs_v2_default_params_50pct": {
        "v1_acc_mean": float(v1_acc.mean()),
        "v1_acc_std": float(v1_acc.std()),
        "v2_acc_mean": float(v2_acc.mean()),
        "v2_acc_std": float(v2_acc.std()),
        "delta": float(v2_acc.mean() - v1_acc.mean()),
    },
    "param_sweep_v2_feats_clip_50pct": {
        "k_list": K_LIST,
        "sigma_list": SIGMA_LIST,
        "acc_grid": acc_grid.tolist(),
        "f1_grid": f1_grid.tolist(),
        "gap_grid": gap_grid.tolist(),
        "n_components_pre_fix": ncomp_pre.tolist(),
    },
    "best_v2_params": {"k": int(best_k_v2), "sigma": best_sigma_v2,
                        "acc": best_acc_v2},
    "label_sweep_v2_feats_at_best_params": {
        f"{int(f*100)}pct": sweep_v2_best[f] for f in LABEL_FRACS
    },
}
out_json = RES_DIR / "level2_results_v2_fix6.json"
out_json.write_text(json.dumps(results_fix6, indent=2), encoding="utf-8")
print("wrote", out_json)
